In [ ]:
import os
os.system("fuser -k 8000/tcp")
print("포트 8000 정리 완료")

In [3]:
!pip install diffusers transformers accelerate safetensors fastapi uvicorn pyngrok python-multipart torch

In [ ]:
# ──────────────────────────────────────────────
# 셀 2: SDXL Base 단일 모델 (메모리 최적화 + 고품질)
# ──────────────────────────────────────────────
import os, io, base64, threading
import torch
from PIL import Image
from diffusers import StableDiffusionXLImg2ImgPipeline, DPMSolverMultistepScheduler
from flask import Flask, request, jsonify

os.system("fuser -k 8000/tcp 2>/dev/null; sleep 1")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("모델 로딩 중...")
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)
pipe.enable_model_cpu_offload()   # VRAM 절약
pipe.enable_attention_slicing(1)  # 메모리 추가 절약
pipe.vae.enable_tiling()          # VAE OOM 방지
print("✅ 모델 로드 완료")

app = Flask(__name__)

NEGATIVE_PROMPT = (
    "multiple logos, grid, collage, frames, watermark, signature, text, letters, "
    "blurry, low quality, distorted, noisy, grainy, pixelated, "
    "multiple images, side by side, tiled, mosaic, panel, border, "
    "background pattern, shadow, gradient background, ugly, deformed"
)

@app.route("/generate", methods=["POST"])
def generate():
    try:
        prompt     = request.form.get("prompt", "a professional logo")
        num_images = int(request.form.get("num_images", 3))
        image_b64  = request.form.get("image_b64", None)

        torch.cuda.empty_cache()

        if image_b64:
            img_bytes  = base64.b64decode(image_b64)
            init_image = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((768, 768))
        else:
            init_image = Image.new("RGB", (768, 768), (255, 255, 255))

        results = []
        for i in range(num_images):
            print(f"[{i+1}/{num_images}] 생성 중...")
            torch.cuda.empty_cache()

            output = pipe(
                prompt=prompt,
                image=init_image,
                strength=0.55,
                guidance_scale=8.0,
                num_inference_steps=30,
                negative_prompt=NEGATIVE_PROMPT,
            ).images[0]

            buf = io.BytesIO()
            output.save(buf, format="PNG")
            results.append(base64.b64encode(buf.getvalue()).decode("utf-8"))
            print(f"[{i+1}/{num_images}] ✅ 완료")

        return jsonify({"images_b64": results})

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

t = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8000))
t.daemon = True
t.start()
print("🚀 Flask 서버 실행 중 (port 8000)")

In [ ]:
# 셀 3: ngrok 터널
from pyngrok import ngrok
import time, requests

ngrok.set_auth_token("")  # ← 토큰은 여기에만

time.sleep(5)  # Flask 서버 시작 대기

# public_url = ngrok.connect(8000, domain = "marklab.ngrok.dev").public_url
public_url = ngrok.connect(8000).public_url
print(f"✅ ngrok URL: {public_url}")
print(f"👉 .env의 COLAB_SD_URL 에 이 값을 붙여넣으세요:\n   {public_url}")

# 헬스체크
try:
    r = requests.get(f"{public_url}/health", timeout=10)
    print(f"서버 상태: {r.json()}")
except Exception as e:
    print(f"헬스체크 실패 (서버 아직 시작 중일 수 있음): {e}")

INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 07:11:15] "GET /health HTTP/1.1" 200 -


✅ ngrok URL: https://unspoken-gloss-pants.ngrok-free.dev
👉 .env의 COLAB_SD_URL 에 이 값을 붙여넣으세요:
   https://unspoken-gloss-pants.ngrok-free.dev
서버 상태: {'status': 'ok'}


In [4]:
# 셀 4: Flask 로컬 헬스체크
import requests

try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print("✅ Flask 응답:", r.json())
except Exception as e:
    print("❌ Flask 응답 없음:", e)

INFO:werkzeug:127.0.0.1 - - [09/Jun/2026 05:13:30] "GET /health HTTP/1.1" 200 -


✅ Flask 응답: {'status': 'ok'}


In [ ]:
import requests
try:
    r = requests.post("http://localhost:8000/generate",
                      data={"prompt": "test logo", "num_images": 1},
                      timeout=120)
    print("상태:", r.status_code)
    print("응답:", r.text[:1000])
except Exception as e:
    print("에러:", e)

[1/1] Base 생성 중...


Traceback (most recent call last):
  File "/tmp/ipykernel_56821/2996705014.py", line 67, in generate
    base_output = base(
                  ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl_img2img.py", line 1284, in __call__
    latents = self.prepare_latents(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl_img2img.py", line 738, in prepare_latents
    init_latents = retrieve_latents(self.vae.encode(image), generator=generator)
                                    ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/diffusers/utils/accelerate_utils.py", line 46, in wrapper
    return method(self, *args, **kwargs)
       

상태: 500
응답: {"error":"CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 149.81 MiB is free. Including non-PyTorch memory, this process has 14.41 GiB memory in use. Of the allocated memory 13.95 GiB is allocated by PyTorch, and 341.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)"}

